## 1. Current work directory

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys, numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__, np.__file__)

## 2. Import packages and define input/output path 

In [ ]:
import glob
import os
import time
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from interface_analyzer import analyze_cfm, plot_cfm_k2_single
from interface_analyzer import PTMModifier, analyze_by_custom_modifier, CSPModifier

SAVE_DIR = LOCAL_OUTPUT_DIR

## 3. Postprocess CFG files, output results.
### Output Data Structure (`cfg_post.pkl`)

The `cfg_post.pkl` file stores the intermediate results generated by the parallel processing script (`Process.py`). This file is the primary input for the Capillary Fluctuation Method (CFM) analysis (`analyze_cfm_ptm`).

The file contains a dictionary where keys correspond to the frame ID (or snapshot number) of the molecular dynamics configuration. Each value is a dictionary containing the extracted physical data needed for fluctuation analysis:

| Data Field | Description | Importance for CFM |
| :--- | :--- | :--- |
| **`h_upper`** & **`h_lower`** | The 1D arrays defining the height profile $h(x)$ of the solid-liquid interfaces (upper and lower), determined using the Brown maximization method on the binned order parameter data. | **CRITICAL:** These arrays are Fourier-transformed to calculate the mean-squared amplitude $\langle|A(k)|^2\rangle$. |
| **`cell`** | The periodic boundary condition (PBC) matrix from OVITO, containing the box dimensions ($L_x, L_y, L_z$) required for calculating the wave vectors ($k$) and the prefactor $k_B T / (L_x L_y)$. | **CRITICAL:** Provides the necessary geometric context for the analysis. |
| **`x`**, **`z`** | Coordinates of the bin centers along the $x$ and $z$ directions, respectively. | Used for spatial reference and calculating bin width/FFT grid setup. |
| **`M`** | The raw 2D binned data of the order parameter (e.g., Centrosymmetry or PTM Solid Flag) used to identify the interface location. | Useful for debugging and verifying the quality of the phase identification. |

The downstream function `analyze_cfm_ptm` reads these dictionary values, aggregates the `h_upper` and `h_lower` profiles across all snapshots, and performs the Fourier analysis.

## 4. Analysis results and plot k and k^2
### Function: `plot_cfm_k2_single(filename, ...)`

This function is designed to analyze the final Capillary Fluctuation Method (CFM) data (which should be stored in a `.dat` file containing $k^2$ vs $k_B T / (L_x L_y \langle|A(k)|^2\rangle)$). The slope of this linear fit directly relates to the interface stiffness ($\tilde{\gamma}$).

#### Key Functionality:

1.  **Optimal Range Selection:** The function iteratively fits the low-$k^2$ data points and selects the subset of points ($n$) that yields the **maximum Coefficient of Determination ($R^2$)**. This provides an objective measure for determining the most linear region of the CFM spectrum.
2.  **Linear Fitting:** Performs a linear fit ($y = m x + b$ or $y = m x$ if `through_origin=True`) on the selected data range.
3.  **Visualization:** Generates a plot showing all data points, highlighting the points used for the best fit, and displaying the resulting fit line.

#### Key Parameters:

| Parameter | Description |
| :--- | :--- |
| **`filename`** | Path to the input `.dat` file (must contain $k^2$, $\text{Ak}_{min}$, and $\text{Ak}_{max}$). |
| **`k2_min`** | Minimum $k^2$ value to consider for the fit. Filters out the first point which is usually zero. |
| **`min_points`** | Minimum number of data points required to perform the linear fit. |
| **`L_min_interface`** | Defines the maximum $k^2$ cutoff based on the expected minimum interface width ($\sim (2\pi / (L_{\text{min}} \cdot a))^2$). |
| **`through_origin`** | If `True`, forces the linear regression line to pass through the origin ($b=0$). |

#### Return Value:

Returns a dictionary containing the calculated fit results, including the calculated `slope` (interface stiffness), `intercept`, maximum `r2`, and the $k^2$ range (`k2_min_used`, `k2_max_used`).

In [ ]:
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# User settings
# =========================
SAVE_DIR = FULL_DATA_ROOT / "100_010_PTM_smoothing_parameter"
TEMPERATURE_K = 925.08
LATTICE_CONST_A = 4.045

# fitting settings
K2_MIN = 5.0e-3
L_MIN_INTERFACE = 4
MIN_POINTS = 6
THROUGH_ORIGIN = True

# =========================
# Helper: parse a_grid and d from filename
# Example:
# 100_010_cfg_post_orientation_grid_2_5_ang_part_d_4_0.pkl
# =========================
def parse_params_from_name(path_obj):
    name = path_obj.name
    m = re.match(
        r"cfg_post_PTM_grid_(\d+_\d+)_d_(\d+_\d+)\.pkl",
        name
    )
    if not m:
        return None, None
    a_grid = float(m.group(1).replace("_", "."))
    d = float(m.group(2).replace("_", "."))
    return a_grid, d

def fmt_sig3(x):
    """Format with 3 significant digits."""
    return f"{x:.3g}"

# =========================
# Collect all pkl files
# =========================
pkl_files = sorted(
    SAVE_DIR.glob("cfg_post_PTM_grid_*_d_*.pkl")
)

if not pkl_files:
    raise FileNotFoundError("No matching pkl files found in current directory.")

summary_rows = []

for pkl_path in pkl_files:
    a_grid, d = parse_params_from_name(pkl_path)
    if a_grid is None:
        print(f"Skipping unrecognized filename: {pkl_path.name}")
        continue

    tag = f"grid_{a_grid:.1f}_d_{d:.1f}".replace(".", "_")
    output_base = SAVE_DIR / f"ptm_cfm_output_{tag}"

    print(f"\n=== Processing {pkl_path.name} ===")
    print(f"a_grid = {a_grid:.1f} Å, d = {d:.1f} Å")

    # -------------------------
    # Step 1: analyze CFM
    # -------------------------
    results_ptm = analyze_cfm(
        pickle_path=pkl_path,
        T=TEMPERATURE_K,
        a=LATTICE_CONST_A,
        use_pchip=True,
        pchipres=1000,
        show_plot=True
    )

    # Save analyze_cfm figure if generated
    plt.savefig(str(output_base) + "_analyze_cfm.png", dpi=300, bbox_inches="tight")
    plt.close()

    # -------------------------
    # Step 2: save k^2 data
    # -------------------------
    k2 = results_ptm["k2"]
    Ak_min = results_ptm["Ak_min"]
    Ak_max = results_ptm["Ak_max"]

    intdata2 = np.c_[k2, Ak_min, Ak_max]

    k2_dat_path = str(output_base) + "_k2.dat"
    np.savetxt(
        k2_dat_path,
        intdata2,
        fmt="%.8e",
        header="k^2 Ak_min Ak_max"
    )
    print(f"CFM data saved to: {k2_dat_path}")

    # -------------------------
    # Step 3: linear fit
    # -------------------------
    res_fit = plot_cfm_k2_single(
        k2_dat_path,
        label=f"a_grid={a_grid:.1f} Å, d={d:.1f} Å",
        k2_min=K2_MIN,
        L_min_interface=L_MIN_INTERFACE,
        min_points=MIN_POINTS,
        through_origin=THROUGH_ORIGIN
    )

    # Save linear fit plot
    plt.savefig(str(output_base) + "_fit.png", dpi=300, bbox_inches="tight")
    plt.close()

    slope = res_fit["slope"]
    slope_sig3 = fmt_sig3(slope)

    print("--- Linear Fit Results ---")
    print(res_fit)
    print(f"slope (3 sig. figs.) = {slope_sig3}")

    summary_rows.append({
        "pkl_file": pkl_path.name,
        "a_grid_A": a_grid,
        "d_A": d,
        "slope": slope,
        "slope_3sig": slope_sig3
    })

# =========================
# Save summary
# =========================
df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values(by=["a_grid_A", "d_A"]).reset_index(drop=True)

summary_csv = SAVE_DIR / "PTM_smoothing_cfm_slope_summary.csv"
df_summary.to_csv(summary_csv, index=False)

print("\n===================================")
print("All jobs finished.")
print(f"Summary saved to: {summary_csv}")
print(df_summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 取数据
df = df_summary.copy()

# 按 a_grid 分组
a_grids = sorted(df["a_grid_A"].unique())

# 定义颜色和 marker
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
markers = ["o", "s", "^", "D","<"]

plt.figure(figsize=(7, 5))

for i, a in enumerate(a_grids):
    sub = df[df["a_grid_A"] == a].sort_values("d_A")

    d_vals = sub["d_A"].values
    slopes = sub["slope"].values

    plt.plot(
        d_vals,
        slopes,
        linestyle="-",
        linewidth=1.8,
        color=colors[i % len(colors)],
        marker=markers[i % len(markers)],
        markersize=7,
        label=f"a_grid = {a:.1f} Å"
    )

plt.xlabel("Smoothing radius d (Å)")
plt.ylabel("Interfacial stiffness (slope)")
plt.title("Sensitivity of interfacial stiffness to smoothing radius")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("stiffness_vs_d_ptm.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np

# ---- 数据 ----
bin_size = np.array([1.5, 2.0, 2.5, 3.0, 3.5])
stiffness = np.array([1.22e-19, 6.23e-19, 7.04e-19, 6.96e-19, 7.21e-19])

# ---- 画图 ----
plt.figure(figsize=(7, 5))

plt.plot(
    bin_size,
    stiffness,
    linestyle="-",
    linewidth=1.8,
    color="tab:blue",
    marker="o",
    markersize=7,
    label="PTM bin averaging"
)

# ---- 坐标轴 ----
plt.xlabel("Bin size (Å)")
plt.ylabel("Interfacial stiffness (slope)")
plt.title("Sensitivity of interfacial stiffness to bin size")

# ---- y轴范围 ----
plt.ylim(0.5e-18, 0.8e-18)

# ---- 图例 & 网格 ----
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()

# ---- 保存 ----
plt.savefig("stiffness_vs_bin_size.png", dpi=300)
plt.show()

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl_d4, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
frames = sorted(results_all.keys())
mean_upper = [results_all[i]["h_upper"].mean() for i in frames]
mean_lower = [results_all[i]["h_lower"].mean() for i in frames]
solid = np.asarray(mean_upper) - np.asarray(mean_lower)

# 画图
plt.figure(figsize=(6,4))
plt.plot(frames, solid, 'o-', lw=1.5, markersize=4)
plt.xlabel("Frame index (timestep)")
plt.ylabel("Solid phase length (Å)")
plt.title("Evolution of solid-liquid interface position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(solid,bins=20,label="100_010")
plt.title("Histogram of Solid Volume")
plt.legend()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl_d8, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
frames = sorted(results_all.keys())
mean_upper = [results_all[i]["h_upper"].mean() for i in frames]
mean_lower = [results_all[i]["h_lower"].mean() for i in frames]
solid = np.asarray(mean_upper) - np.asarray(mean_lower)

# 画图
plt.figure(figsize=(6,4))
plt.plot(frames, solid, 'o-', lw=1.5, markersize=4)
plt.xlabel("Frame index (timestep)")
plt.ylabel("Solid phase length (Å)")
plt.title("Evolution of solid-liquid interface position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(solid,bins=20,label="100_010")
plt.title("Histogram of Solid Volume")
plt.legend()
plt.show()

In [ ]:
first_key = list(results_full.keys())[0]
print(f"Key type: {type(first_key)}")
print(f"Total keys: {len(results_full.keys())}")

if isinstance(results_full[first_key], (list, np.ndarray)):
    print(f"Value length: {len(results_full[first_key])}")

In [ ]:
type(results_all)

In [ ]:
from matplotlib import pyplot as plt
from pathlib import Path
import pickle
SAVE_DIR = FULL_DATA_ROOT
Path_ptm_pkl_a_2p5_d_6 = SAVE_DIR / "cfg_post_PTM_grid_2_5_d_6_0.pkl"
with open(Path_ptm_pkl_a_2p5_d_6, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
Phase = results_all[5000000]["M"]

plt.figure(figsize=(10, 8))
plt.imshow(Phase, cmap='viridis', origin='lower')
plt.colorbar(label='Phase Value', shrink=0.5)
plt.xlabel('X index')
plt.ylabel('Y index')
#plt.savefig("phase_heatmap_PTM_d4.png")
plt.show()

In [ ]:
Boundary = results_all[5000000]["h_lower"]
plt.figure(figsize=(8, 2.0))
plt.plot(Boundary)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.show()